# 02 — Bagging と Random Forest
## なぜ「平均」で強くなるのか
独立で同分散 $\sigma^2$ の推定量 $T_1,\dots,T_B$ を平均すると

$$\mathrm{Var}\Big(\frac1B\sum_b T_b\Big)=\frac{\sigma^2}{B}.$$

実際の木は相関する。全ての木の対の相関係数が $\rho$ なら

$$\mathrm{Var}(\bar T)=\sigma^2\left[\rho+\frac{1-\rho}{B}\right].$$

つまり本数 $B$ を増やすだけでなく、**相関 $\rho$ を減らす** のが重要。

- **Bagging:** 学習データを復元抽出 (bootstrap) して各モデルを訓練、回帰では平均を取る。
- **Random Forest:** Bagging に加え、通常は**各分割候補で特徴量をランダムに選ぶ**（特徴量サブサンプリング）ため、木の相関を減らせる。今回は bootstrap だけを NumPy で実装する。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
rng = np.random.default_rng(42)
x = np.sort(rng.uniform(0, 1, 90))
y = np.sin(2*np.pi*x) + rng.normal(0,.25,len(x))
grid = np.linspace(0,1,200).reshape(-1,1)
fig,axes=plt.subplots(1,2,figsize=(12,3.5))
for seed in range(8):
    inds = np.random.default_rng(seed).integers(0,len(x),len(x))
    tree = DecisionTreeRegressor(max_depth=5,random_state=seed).fit(x[inds,None], y[inds])
    axes[0].plot(grid[:,0],tree.predict(grid),alpha=.45)
axes[0].scatter(x,y,s=8,color="black")
axes[0].set_title("Bootstrap trees vary")
rf = RandomForestRegressor(n_estimators=100,max_depth=5,random_state=4,n_jobs=2)
rf.fit(x[:,None],y)
axes[1].scatter(x,y,s=8,color="black")
axes[1].plot(grid[:,0],rf.predict(grid),linewidth=2)
axes[1].set_title("Forest average")
plt.tight_layout(); plt.show()

## 演習
\`bootstrap_indices(n_samples, rng)\` を書く。長さ $n$ の index array を、重複を許す復元抽出で生成すること。渡された \`np.random.Generator\` を必ず使い、再現性を確保する。

**確認:** 100個を復元抽出したとき、おおよそ何種類の元のデータが1回以上選ばれるか？ 期待値は $n[1-(1-1/n)^n]\approx0.632n$。

In [ ]:
n = 100
expected_unique = n*(1-(1-1/n)**n)
print("Expected unique observations:",round(expected_unique,1))
print('採点: python tools/grade.py -k test_bootstrap')

## Bagging と Boosting は何が違う？
Bagging は木を概ね **並列** に育てて平均。Boosting は前のモデルの失敗を **逐次的** に補う。従って目的と学習ダイナミクスが異なる。